In [36]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

CSV_PATH = "../data/fuel_prices.csv"  # Change if needed
BATCH_SIZE = 200

df = pd.read_csv(CSV_PATH)

print(df.shape)
df.head()


# Ensure required columns exist
for col in ["Address", "City", "State"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

for col in ["latitude", "longitude"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "geocode_status" not in df.columns:
    df["geocode_status"] = pd.NA

df["geocode_status"] = df["geocode_status"].astype("string")

# A normalized key identifies duplicate locations.
def make_location_key(row):
    return "|".join(
        str(row[col]).strip().casefold()
        for col in ["Address", "City", "State"]
    )

df["_location_key"] = df.apply(make_location_key, axis=1)

# Reuse coordinates already stored for the same location.
for key, group in df.groupby("_location_key"):
    valid = group[
        group["latitude"].notna()
        & group["longitude"].notna()
    ]

    if not valid.empty:
        lat = valid.iloc[0]["latitude"]
        lon = valid.iloc[0]["longitude"]

        missing_coords = (
            df["_location_key"].eq(key)
            & (df["latitude"].isna() | df["longitude"].isna())
        )

        df.loc[missing_coords, "latitude"] = lat
        df.loc[missing_coords, "longitude"] = lon
        df.loc[missing_coords, "geocode_status"] = "success"

geolocator = Nominatim(
    user_agent="fuel_route_assessment_v1",
    timeout=15
)

geocode_limited = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    max_retries=1,
    error_wait_seconds=2,
    swallow_exceptions=True
)

def save_progress():
    # Do not save the temporary matching key.
    df.drop(columns=["_location_key"], errors="ignore").to_csv(
        CSV_PATH, index=False
    )

# Process only locations that have no coordinates and haven't
# already been marked not_found.
pending_mask = (
    df["latitude"].isna()
    & df["longitude"].isna()
    & ~df["geocode_status"].isin(["not_found"])
)

pending = df.loc[pending_mask].drop_duplicates(
    subset=["_location_key"]
).head(BATCH_SIZE)

print(f"Unique locations in this batch: {len(pending)}")

for idx, row in pending.iterrows():
    query = (
        f"{row['Address']}, {row['City']}, "
        f"{row['State']}, USA"
    )

    try:
        location = geocode_limited(query)

        if location:
            lat, lon = location.latitude, location.longitude

            same_location = df["_location_key"].eq(
                row["_location_key"]
            )

            df.loc[same_location, "latitude"] = lat
            df.loc[same_location, "longitude"] = lon
            df.loc[same_location, "geocode_status"] = "success"

            print(
                f"SUCCESS: {query} -> "
                f"({lat:.6f}, {lon:.6f})"
            )
        else:
            df.loc[
                df["_location_key"].eq(row["_location_key"]),
                "geocode_status"
            ] = "not_found"

            print(f"NOT FOUND: {query}")

    except Exception as exc:
        df.loc[
            df["_location_key"].eq(row["_location_key"]),
            "geocode_status"
        ] = "error"

        print(f"ERROR: {query}: {exc}")

    # Persist each lookup so progress survives interruptions.
    save_progress()

print("\nBatch saved to:", CSV_PATH)
print("Successful rows:", (df["geocode_status"] == "success").sum())
print("Not found:", (df["geocode_status"] == "not_found").sum())
print("Errors:", (df["geocode_status"] == "error").sum())
print(
    "Remaining rows without coordinates:",
    (df["latitude"].isna() | df["longitude"].isna()).sum()
)

df.drop(columns=["_location_key"], errors="ignore", inplace=True)

(8151, 10)
Unique locations in this batch: 200
NOT FOUND: I-75, EXIT 49, East Bernstadt, KY, USA
NOT FOUND: I-70, EXIT 3, Hancock, MD, USA
NOT FOUND: I-55,  EXIT 40, Marston, MO, USA
NOT FOUND: I-70, EXIT 359 & US-24, Limon, CO, USA
NOT FOUND: I-77, EXIT 100 & 101, Mount Airy, NC, USA
NOT FOUND: I-57/I-64, EXIT 95 & SR-15, Mount Vernon, IL, USA
NOT FOUND: I-30, EXIT 63 & SR-53, Gurdon, AR, USA
NOT FOUND: I-70 EXIT 121 & US-40, Columbia, MO, USA
SUCCESS: US-60, Norwood, MO, USA -> (37.110948, -92.452098)
NOT FOUND: I-12, EXIT 40 & US-51 BUS, Hammond, LA, USA
SUCCESS: US-30, Columbus, NE, USA -> (41.424305, -97.368212)
NOT FOUND: I-20, EXIT 71 & US-21, Columbia, SC, USA
NOT FOUND: I-10, EXIT 36 & SR-397, Lake Charles, LA, USA
SUCCESS: I-24, EXIT 16, Paducah, KY, USA -> (37.093960, -88.686735)
NOT FOUND: I-77, EXIT 111, North Canton, OH, USA
NOT FOUND: I-94, EXIT 147, Mandan, ND, USA
NOT FOUND: I-45 & FM 1603, EXIT 238, Rice, TX, USA
SUCCESS: US-84 & US-171, Mansfield, LA, USA -> (32.0218

In [37]:
# Filter the DataFrame to keep only successful rows
df_success = df[df["geocode_status"] == "success"]
print(df_success.shape)
df_success.head()

(216, 10)


,OPIS Truckstop ID,Truckstop Name,Address,City,State,Rack ID,Retail Price,latitude,longitude,geocode_status
5,35,ACI TRUCK STOP,US-46,Columbia,NJ,90,3.079,40.927229,-75.088798,success
12,75,Good 2 Go #709,US-50,Montrose,CO,810,3.424,38.479856,-107.879058,success
13,77,OASIS TRAVEL PLAZA,US-13,Laurel,DE,243,3.229,38.571711,-75.563410,success
14,79,DELAWARE TRUCK PLAZA,US-13 & US-40,New Castle,DE,243,3.249,39.615985,-75.662956,success
26,127,CIRCLE K #4700095,US-41,Henderson,KY,340,3.609,37.904393,-87.550779,success


# PostgreSQL Database

In [42]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Database configuration
DB_NAME = os.getenv("DB_NAME", "strata_fuel_db")
DB_USER = os.getenv("DB_USER", "strata_fuel_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "StrataFuelEngine")
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", 5432)

TABLE_NAME = "fuel_stations"
BATCH_SIZE = 100

# Create PostgreSQL connection
DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

# Read all station records from PostgreSQL
with engine.connect() as conn:
    df = pd.read_sql(
        text(f'SELECT * FROM "{TABLE_NAME}"'),
        conn
    )

print("Data shape:", df.shape)
display(df.head())

# Validate required columns
required_columns = ["address", "city", "state"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Normalize location fields
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Ensure coordinate and status columns exist in the DataFrame
for col in ["latitude", "longitude"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "geocode_status" not in df.columns:
    df["geocode_status"] = pd.NA

df["geocode_status"] = df["geocode_status"].astype("string")

# A normalized key identifies duplicate locations.
def make_location_key(row):
    return "|".join(
        str(row[col]).strip().casefold()
        for col in ["address", "city", "state"]
    )

df["_location_key"] = df.apply(make_location_key, axis=1)

# Reuse existing coordinates for duplicate locations.
for key, group in df.groupby("_location_key"):
    valid = group[
        group["latitude"].notna()
        & group["longitude"].notna()
    ]

    if not valid.empty:
        lat = valid.iloc[0]["latitude"]
        lon = valid.iloc[0]["longitude"]

        missing_coords = (
            df["_location_key"].eq(key)
            & (df["latitude"].isna() | df["longitude"].isna())
        )

        df.loc[missing_coords, "latitude"] = lat
        df.loc[missing_coords, "longitude"] = lon
        df.loc[missing_coords, "geocode_status"] = "success"

# Initialize geocoder with a conservative request rate.
geolocator = Nominatim(
    user_agent="fuel_route_assessment_v1",
    timeout=15
)

geocode_limited = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    max_retries=1,
    error_wait_seconds=2,
    swallow_exceptions=True
)

# Identify the primary key used to update individual database records.
# Change this if your table uses a different primary-key column.
PRIMARY_KEY = "id"

if PRIMARY_KEY not in df.columns:
    raise ValueError(
        f"Primary key column '{PRIMARY_KEY}' not found. "
        f"Available columns: {df.columns.tolist()}. "
        "Set PRIMARY_KEY to your table's actual primary-key column."
    )

# Persist coordinates/status for all rows sharing a location.
def save_location_progress(location_key):
    matching_rows = df[df["_location_key"].eq(location_key)]

    with engine.begin() as conn:
        for _, record in matching_rows.iterrows():
            conn.execute(
                text(f"""
                    UPDATE "{TABLE_NAME}"
                    SET latitude = :latitude,
                        longitude = :longitude,
                        geocode_status = :geocode_status
                    WHERE "{PRIMARY_KEY}" = :record_id
                """),
                {
                    "latitude": (
                        None if pd.isna(record["latitude"])
                        else float(record["latitude"])
                    ),
                    "longitude": (
                        None if pd.isna(record["longitude"])
                        else float(record["longitude"])
                    ),
                    "geocode_status": (
                        None if pd.isna(record["geocode_status"])
                        else str(record["geocode_status"])
                    ),
                    "record_id": record[PRIMARY_KEY]
                }
            )

# Process only locations that need geocoding.
pending_mask = (
    (df["latitude"].isna() | df["longitude"].isna())
    & ~df["geocode_status"].isin(["not_found"])
)

pending = df.loc[pending_mask].drop_duplicates(
    subset=["_location_key"]
).head(BATCH_SIZE)

print(f"Unique locations in this batch: {len(pending)}")

for _, row in pending.iterrows():
    query = (
        f"{row['address']}, {row['city']}, "
        f"{row['state']}, USA"
    )
    location_key = row["_location_key"]

    try:
        location = geocode_limited(query)

        if location:
            lat, lon = location.latitude, location.longitude
            same_location = df["_location_key"].eq(location_key)

            df.loc[same_location, "latitude"] = lat
            df.loc[same_location, "longitude"] = lon
            df.loc[same_location, "geocode_status"] = "success"

            print(
                f"SUCCESS: {query} -> ({lat:.6f}, {lon:.6f})"
            )
        else:
            df.loc[
                df["_location_key"].eq(location_key),
                "geocode_status"
            ] = "not_found"

            print(f"NOT FOUND: {query}")

    except Exception as exc:
        df.loc[
            df["_location_key"].eq(location_key),
            "geocode_status"
        ] = "error"

        print(f"ERROR: {query}: {exc}")

    # Save each lookup to PostgreSQL immediately.
    save_location_progress(location_key)

print("\nBatch saved to PostgreSQL table:", TABLE_NAME)
print("Successful rows:", (df["geocode_status"] == "success").sum())
print("Not found:", (df["geocode_status"] == "not_found").sum())
print("Errors:", (df["geocode_status"] == "error").sum())
print(
    "Remaining rows without coordinates:",
    (df["latitude"].isna() | df["longitude"].isna()).sum()
)

# Clean up the temporary matching key.
df.drop(columns=["_location_key"], errors="ignore", inplace=True)

engine.dispose()

Data shape: (6967, 13)


,id,opis_truckstop_id,name,address,city,state,rack_id,retail_price,latitude,longitude,created_at,updated_at,geocode_status
0,1297,20011,CUBBYS #111,US-81,Norfolk,NE,470,3.20900,42.008814,-97.426517,2026-10-17 10:47:37.242791+00:00,2026-10-17 19:39:54.442328+00:00,success
1,4619,69875,LOVES TRAVEL STOP #680,US-81,Norfolk,NE,470,3.31567,42.008814,-97.426517,2026-10-17 10:47:37.372016+00:00,2026-10-17 19:39:54.618062+00:00,success
2,2640,65683,ALLSUPS CONVENIENCE STORE #2225,US-60,Friona,TX,585,2.99900,34.634288,-102.738575,2026-10-17 10:47:37.285262+00:00,2026-10-17 19:39:54.491297+00:00,success
3,1191,9340,LANCASTER TRAVEL PLAZA,US-30,Ronks,PA,208,3.89900,40.019717,-76.178888,2026-10-17 10:47:37.242614+00:00,2026-10-17 19:39:54.442170+00:00,success
4,1306,20036,SMOKIN JOES,I-90 EXIT 24,Sanborn,NY,38,3.69900,NaN,NaN,2026-10-17 10:47:37.242806+00:00,2026-10-17 19:39:54.442342+00:00,not_found


Unique locations in this batch: 100
SUCCESS: US-41, Little Chute, WI, USA -> (44.297891, -88.349677)
NOT FOUND: I-95, EXIT 89 & SR-802, Ashland, VA, USA
NOT FOUND: US-6/CR-74, Hastings, NE, USA
NOT FOUND: I-10, EXIT 37, El Paso, TX, USA
NOT FOUND: I-10, EXIT 68, Deming, NM, USA
NOT FOUND: US-20, EXIT 11 & SR-140, Shrewsbury, MA, USA
NOT FOUND: I-80, EXIT 237 & US-283, Lexington, NE, USA
NOT FOUND: I-25, EXIT 135, Douglas, WY, USA
NOT FOUND: I-80, EXIT 19 & SR-659, Sparks, NV, USA
NOT FOUND: I-30, EXIT 106, Malvern, AR, USA
NOT FOUND: I-70, EXIT 11 & CR-41, Valley Grove, WV, USA
SUCCESS: US- 22 & SR-322, Duncannon, PA, USA -> (40.393143, -77.032575)
NOT FOUND: I-77, EXIT 95 & SR-61, Charleston, WV, USA
NOT FOUND: I-70, EXIT 47, Palisade, CO, USA
SUCCESS: US-20 & US-83, Valentine, NE, USA -> (42.874675, -100.550616)
SUCCESS: US-275, Wisner, NE, USA -> (41.985189, -96.910465)
NOT FOUND: I-95, EXIT 3, Portsmouth, NH, USA
NOT FOUND: I-85, EXIT 42, Tuskegee, AL, USA
NOT FOUND: I-82, EXIT 1 &

# SQLite3 Database

In [44]:
import sqlite3
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# SQLite database configuration
DB_PATH = "../db.sqlite3"
TABLE_NAME = "fuel_stations"
PRIMARY_KEY = "id"
BATCH_SIZE = 1000

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)

# Read station records
df = pd.read_sql_query(
    f'SELECT * FROM "{TABLE_NAME}"',
    conn
)

print("Data shape:", df.shape)
display(df.head())

# Validate required columns
required_columns = ["address", "city", "state"]
missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    conn.close()
    raise ValueError(f"Missing required columns: {missing_columns}")

# Ensure coordinate columns exist in SQLite
existing_columns = {
    row[1]
    for row in conn.execute(f'PRAGMA table_info("{TABLE_NAME}")')
}

for col in ["latitude", "longitude", "geocode_status"]:
    if col not in existing_columns:
        column_type = (
            "REAL" if col in ["latitude", "longitude"]
            else "TEXT"
        )
        conn.execute(
            f'ALTER TABLE "{TABLE_NAME}" '
            f'ADD COLUMN "{col}" {column_type}'
        )

conn.commit()

# Normalize location fields
for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

for col in ["latitude", "longitude"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "geocode_status" not in df.columns:
    df["geocode_status"] = pd.NA

df["geocode_status"] = df["geocode_status"].astype("string")

# Verify the primary key
if PRIMARY_KEY not in df.columns:
    conn.close()
    raise ValueError(
        f"Primary key '{PRIMARY_KEY}' not found. "
        f"Available columns: {df.columns.tolist()}. "
        "Change PRIMARY_KEY to the actual unique key."
    )

if df[PRIMARY_KEY].isna().any() or df[PRIMARY_KEY].duplicated().any():
    conn.close()
    raise ValueError(
        f"'{PRIMARY_KEY}' must uniquely identify every database row."
    )

# Build normalized keys for duplicate locations
def make_location_key(row):
    return "|".join(
        str(row[col]).strip().casefold()
        for col in ["address", "city", "state"]
    )

df["_location_key"] = df.apply(make_location_key, axis=1)

# Reuse coordinates already available for duplicate locations
for key, group in df.groupby("_location_key"):
    valid = group[
        group["latitude"].notna()
        & group["longitude"].notna()
    ]

    if not valid.empty:
        lat = valid.iloc[0]["latitude"]
        lon = valid.iloc[0]["longitude"]

        missing_coords = (
            df["_location_key"].eq(key)
            & (df["latitude"].isna() | df["longitude"].isna())
        )

        df.loc[missing_coords, "latitude"] = lat
        df.loc[missing_coords, "longitude"] = lon
        df.loc[missing_coords, "geocode_status"] = "success"

# Geocoder
geolocator = Nominatim(
    user_agent="fuel_route_assessment_v1",
    timeout=15
)

geocode_limited = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    max_retries=1,
    error_wait_seconds=2,
    swallow_exceptions=True
)

# Save all rows matching a location key
def save_location_progress(location_key):
    matching_rows = df[df["_location_key"].eq(location_key)]

    try:
        with conn:
            for _, record in matching_rows.iterrows():
                conn.execute(
                    f"""
                    UPDATE "{TABLE_NAME}"
                    SET latitude = ?,
                        longitude = ?,
                        geocode_status = ?
                    WHERE "{PRIMARY_KEY}" = ?
                    """,
                    (
                        None if pd.isna(record["latitude"])
                        else float(record["latitude"]),
                        None if pd.isna(record["longitude"])
                        else float(record["longitude"]),
                        None if pd.isna(record["geocode_status"])
                        else str(record["geocode_status"]),
                        record[PRIMARY_KEY]
                    )
                )
    except Exception:
        raise

# Select locations that need geocoding
pending_mask = (
    (df["latitude"].isna() | df["longitude"].isna())
    & ~df["geocode_status"].isin(["not_found"])
)

pending = df.loc[pending_mask].drop_duplicates(
    subset=["_location_key"]
).head(BATCH_SIZE)

print(f"Unique locations in this batch: {len(pending)}")

for _, row in pending.iterrows():
    query = (
        f"{row['address']}, {row['city']}, "
        f"{row['state']}, USA"
    )
    location_key = row["_location_key"]
    same_location = df["_location_key"].eq(location_key)

    try:
        location = geocode_limited(query)

        if location:
            lat, lon = location.latitude, location.longitude

            df.loc[same_location, "latitude"] = lat
            df.loc[same_location, "longitude"] = lon
            df.loc[same_location, "geocode_status"] = "success"

            print(
                f"SUCCESS: {query} -> ({lat:.6f}, {lon:.6f})"
            )
        else:
            df.loc[same_location, "geocode_status"] = "not_found"
            print(f"NOT FOUND: {query}")

    except Exception as exc:
        df.loc[same_location, "geocode_status"] = "error"
        print(f"ERROR: {query}: {exc}")

    # Persist every lookup immediately
    save_location_progress(location_key)

print("\nBatch saved to SQLite table:", TABLE_NAME)
print("Successful rows:", (df["geocode_status"] == "success").sum())
print("Not found:", (df["geocode_status"] == "not_found").sum())
print("Errors:", (df["geocode_status"] == "error").sum())
print(
    "Remaining rows without coordinates:",
    (df["latitude"].isna() | df["longitude"].isna()).sum()
)

# Remove the temporary key and close the database connection
df.drop(columns=["_location_key"], errors="ignore", inplace=True)
conn.close()

Data shape: (6967, 13)


,id,opis_truckstop_id,name,address,city,state,rack_id,retail_price,latitude,longitude,created_at,updated_at,geocode_status
0,1,7,WOODSHED OF BIG CABIN,"I-44, EXIT 283 & US-69",Big Cabin,OK,307,3.00733,NaN,NaN,2026-10-17 10:47:37.187000,2026-10-17 19:39:54.370000,not_found
1,2,9,KWIK TRIP #796,"I-94, EXIT 143 & US-12 & SR-21",Tomah,WI,420,3.28733,NaN,NaN,2026-10-17 10:47:37.187000,2026-10-17 19:39:54.370000,not_found
2,3,20,PILOT TRAVEL CENTER #1243,"I-8, EXIT 119 & SR-85",Gila Bend,AZ,930,3.89900,NaN,NaN,2026-10-17 10:47:37.187000,2026-10-17 19:39:54.370000,not_found
3,4,20,PILOT #1243,"I-8, EXIT 119 & SR-85",Gila Bend,AZ,930,3.89900,NaN,NaN,2026-10-17 10:47:37.187000,2026-10-17 19:39:54.370000,not_found
4,5,28,LITTLEFIELD EXPRESS #2,"I-540, EXIT 12 & US-71",Fort Smith,AR,645,3.39900,NaN,NaN,2026-10-17 10:47:37.187000,2026-10-17 19:39:54.370000,not_found


Unique locations in this batch: 1000
NOT FOUND: I-80, EXIT 440 & SR-50, EXIT 440, Omaha, NE, USA
NOT FOUND: I-80, EXIT 312 & US-281/US-34, Grand Island, NE, USA
NOT FOUND: I-87, EXIT 17N & US-9, Fort Edward, NY, USA
NOT FOUND: I-90, EXIT 96 & US-93/SR-200, Missoula, MT, USA
NOT FOUND: SR-17, EXIT 20, Salamanca, NY, USA
NOT FOUND: I-90, EXIT 96 & US-93, Missoula, MT, USA
NOT FOUND: I-90, EXIT 437, Laurel, MT, USA
NOT FOUND: I-44, EXIT 88 & SR-125, Strafford, MO, USA
NOT FOUND: I-70, EXIT 28, Oak Grove, MO, USA
NOT FOUND: I-55, EXIT 58 & SR-80, Matthews, MO, USA
NOT FOUND: I-70, EXIT 148 & US-54, Kingdom City, MO, USA
NOT FOUND: I-44, EXIT 127, Lebanon, MO, USA
NOT FOUND: I-55, EXIT 220, Tillatoba, MS, USA
NOT FOUND: I-70, EXIT 208, Belmont, OH, USA
NOT FOUND: I-20 & I-59, EXIT 160, Meridian, MS, USA
NOT FOUND: I-94/US-52, EXIT 127 & US-71, Sauk Centre, MN, USA
SUCCESS: I-35 & I-90, EXIT 11, Albert Lea, MN, USA -> (43.648013, -93.368266)
NOT FOUND: I-69,EXIT 25 & SR-60, Tekonsha, MI, USA